# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
!pip install datasets

In [8]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

print(ds.features)
print(next(iter(ds)))

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

In [9]:
from itertools import islice
import pandas as pd

rows = list(islice(ds, 100000))
df = pd.DataFrame(rows)
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   report_date               100000 non-null  object 
 1   client_hash_id            100000 non-null  object 
 2   content_hash_id           100000 non-null  object 
 3   client_has_gsc            100000 non-null  bool   
 4   client_has_ga4            100000 non-null  bool   
 5   gsc_data_available        100000 non-null  bool   
 6   ga4_data_available        100000 non-null  bool   
 7   gsc_impressions           100000 non-null  int64  
 8   gsc_clicks                100000 non-null  int64  
 9   gsc_sum_position          99999 non-null   float64
 10  gsc_avg_position          99999 non-null   float64
 11  ga4_pageviews             100000 non-null  int64  
 12  ga4_sessions              100000 non-null  int64  
 13  ga4_users                 100000 non-null  in

In [12]:
df['report_date'] = pd.to_datetime(df['report_date'])
df['month'] = df['report_date'].dt.to_period("M")
df['month'].value_counts()

,count
month,
2025-02,75985
2025-03,22718
2025-01,1297


Unit of analysis

- One row represents the daily performance of one content item for one client on one report date.
- Table used: `fact_content_daily_performance`.
- Time window: **February 2025 (2025-02)** because it is the largest complete month available in this dataset.
- Prediction task: Predict whether a content item will receive higher Google Search Console clicks.
- Excluded: Future click information is excluded because it would leak the target.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

####Features: "gsc_impressions", "gsc_avg_position", "ga4_sessions", "ga4_users", "scroll_events". Because historical information is available before prediction.
####Label: "High GSC clicks" This is a predicted target.
####Context: "report_date", "client_hash_id", "content_hash_id". Because they describe the observations.
####Excluded: "gsc_clicks" and "client_hash_id" because "client_hash_id" is an identifier and does not convey predictive information. Therefore, future clicks would leak the target.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
!pip install -q duckdb

In [23]:
import duckdb

# Create a copy of the DataFrame and drop the 'month' column
df_for_duckdb = df.drop(columns=['month'])

duckdb.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS cnt
FROM df_for_duckdb
GROUP BY 1,2,3
HAVING COUNT(*)>1
""")

┌──────────────┬────────────────┬─────────────────┬───────┐
│ report_date  │ client_hash_id │ content_hash_id │  cnt  │
│ timestamp_ns │    varchar     │     varchar     │ int64 │
├──────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                          │
└─────────────────────────────────────────────────────────┘

####No duplicate rows were found. The grain is one content item per client per day.

In [24]:
duckdb.sql("""
SELECT COUNT(*) AS row_count, MIN(report_date) AS first_day, MAX(report_date) AS last_day
FROM df_for_duckdb
WHERE report_date BETWEEN DATE '2025-02-01' AND DATE '2025-02-28'
""")

┌───────────┬─────────────────────┬─────────────────────┐
│ row_count │      first_day      │      last_day       │
│   int64   │    timestamp_ns     │    timestamp_ns     │
├───────────┼─────────────────────┼─────────────────────┤
│     75985 │ 2025-02-01 00:00:00 │ 2025-02-28 00:00:00 │
└───────────┴─────────────────────┴─────────────────────┘

In [25]:
duckdb.sql("""
SELECT COUNT(*) AS available_rows
FROM df_for_duckdb
WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2025-02-01'
        AND DATE '2025-02-28'
""")

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│          75985 │
└────────────────┘

Feature availability

- gsc_impressions — Knowable at the decision moment because they come from historical Search Console data.
- gsc_avg_position — Known because search ranking already exists before prediction.
- ga4_sessions — Historical sessions are available before prediction.
- ga4_users — Historical users are already observed.
- scroll_events — Historical engagement is measured before prediction.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [26]:
feature_df = df.loc[
    df["month"] == "2025-02",
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_users",
        "scroll_events",
    ],
]

feature_df.head()

,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_users,scroll_events
1297,9,25.555556,0,0,0
1298,5,8.800000,0,0,0
1299,13,42.846154,0,0,0
1300,27,34.148148,0,0,0
1301,6,12.500000,0,0,0


In [27]:
median_clicks = df["gsc_clicks"].median()

df["label"] = (df["gsc_clicks"] > median_clicks).astype(int)

In [28]:
df['label']

,label
0,0
1,0
2,0
3,0
4,0
...,...
99995,0
99996,0
99997,0
99998,1


####Adding "gsc_clicks" as a feature would leak information because the label is directly derived from clicks. This would artificially increase model performance, so the feature must be removed.

####This dataset contains historical Search Console and GA4 metrics only. It cannot explain why users behaved in a certain way or capture future events. Early rows may have incomplete GA4 data because availability changes over time.The available dataset only contains data from January to March 2025, so it cannot capture longer-term seasonal trends.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.